<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 2 — Data Cleaning & Feature Engineering
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | Notebook 1 complété |
| **Niveau** | Avancé |
| **Outils** | Python — pandas, numpy, matplotlib |
| **Durée estimée** | 3h à 4h |

> Le Notebook 1 a identifié **9 anomalies** et établi les KPIs de référence. Ce notebook les corrige toutes systématiquement, construit les variables analytiques clés et exporte la table finale `logitrack_analytics.csv`.

---
## 0. Mise en place de l'environnement

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## Étape 1 — Chargement depuis zéro

### MÉTHODE
On recharge les 7 CSV depuis zéro — état propre et reproductible. Ne jamais continuer depuis l'état du Notebook 1 sans recharger : les opérations interactives d'exploration peuvent avoir modifié des DataFrames en mémoire.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack/data/'

df_liv  = pd.read_csv(BASE_URL + 'livraisons.csv',
                       parse_dates=['date_creation','date_enlevement',
                                    'date_livraison_prevue','date_livraison_reelle'])
df_trp  = pd.read_csv(BASE_URL + 'transporteurs.csv')
df_ent  = pd.read_csv(BASE_URL + 'entrepots.csv')
df_rte  = pd.read_csv(BASE_URL + 'routes.csv')
df_cli  = pd.read_csv(BASE_URL + 'clients.csv')
df_evt  = pd.read_csv(BASE_URL + 'evenements.csv',  parse_dates=['date_evenement'])
df_alt  = pd.read_csv(BASE_URL + 'alertes_sla.csv', parse_dates=['date_alerte'])

print('Chargement OK ✅')
for name, df in [('livraisons',df_liv),('transporteurs',df_trp),('entrepots',df_ent),
                  ('routes',df_rte),('clients',df_cli),('evenements',df_evt),
                  ('alertes_sla',df_alt)]:
    print(f'  {name:<20} {len(df):>7,} lignes x {len(df.columns)} colonnes')

---
## Étape 2 — Nettoyage des anomalies

### MÉTHODE — Ordre de correction
On suit toujours le même ordre :
1. **Doublons** (réduire le volume avant les autres opérations)
2. **Aberrants numériques** (poids négatifs, CSAT hors plage)
3. **Incohérences calculées** (délais aberrants)
4. **Exclusions métier** (transporteur suspendu)

**Règle fondamentale :** on ne supprime jamais de la table source. On applique des filtres et corrections dans `df_liv` tout en conservant les lignes brutes dans `df_liv_raw` pour traçabilité.

In [ ]:
# Sauvegarde de la table brute pour traçabilité
df_liv_raw = df_liv.copy()
print(f'Table brute sauvegardée : {len(df_liv_raw):,} lignes')

### 2.1 — Doublons sur livraison_id

In [ ]:
# TODO : supprimer les doublons sur livraison_id (keep='first')
n_avant = len(df_liv)
df_liv = # TODO
print(f'Doublons supprimés : {n_avant - len(df_liv)}')

### 2.2 — Poids kg aberrants (≤ 0)

> **MÉTHODE — Imputation par médiane du type_colis**
>
> On impute par la médiane du même `type_colis` plutôt que la médiane globale. Un colis Pharmaceutique et un colis Industriel n'ont pas les mêmes distributions de poids — une médiane globale introduirait un biais.

In [ ]:
mask_poids = df_liv['poids_kg'] <= 0
print(f'Poids aberrants : {mask_poids.sum()}')
print(df_liv[mask_poids][['livraison_id','type_colis','poids_kg']])

# TODO : calculer medianes_poids par type_colis (sur poids > 0 uniquement)
medianes_poids = # TODO

# TODO : boucler sur les lignes aberrantes et imputer la médiane du type_colis
for idx in df_liv[mask_poids].index:
    tc = df_liv.loc[idx, 'type_colis']
    df_liv.loc[idx, 'poids_kg'] = # TODO

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 2.3 — CSAT hors plage [1-5]

In [ ]:
mask_csat = df_liv['csat'].notna() & ((df_liv['csat'] < 1) | (df_liv['csat'] > 5))
print(f'CSAT hors plage : {mask_csat.sum()}')

# TODO : remplacer les CSAT hors plage par np.nan
df_liv.loc[mask_csat, 'csat'] = # TODO
print(f'CSAT corrigés : {mask_csat.sum()}')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 2.4 — Délais aberrants (-99j)

> **MÉTHODE — Recalcul depuis les dates sources**
>
> `delai_jours = duree_reelle_jours - sla_jours_contractuel` est une colonne calculée. Quand elle est aberrante, on la recalcule depuis ses composantes — qui, elles, sont fiables. C'est le principe de **dériver le calculé depuis le primitif**.

In [ ]:
mask_delai = df_liv['delai_jours'] < -30
print(f'Délais aberrants : {mask_delai.sum()}')

# TODO : recalculer delai_jours depuis duree_reelle_jours - sla_jours_contractuel
# Indice : opérer uniquement sur les lignes du masque
df_liv.loc[mask_delai, 'delai_jours'] = # TODO

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 2.5 — Exclusion du transporteur suspendu

In [ ]:
trp_actifs = df_trp[df_trp['actif'] == True]['transporteur_id'].tolist()
mask_suspendu = ~df_liv['transporteur_id'].isin(trp_actifs)
print(f'Livraisons avec transporteur suspendu : {mask_suspendu.sum()}')

# TODO : créer colonne transporteur_actif (1 si actif, 0 si suspendu)
# Indice : .isin(trp_actifs).astype(int)
df_liv['transporteur_actif'] = # TODO
print(df_liv['transporteur_actif'].value_counts())

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [ ]:
print('=== BILAN NETTOYAGE ===')
print(f'  Lignes brutes         : {len(df_liv_raw):,}')
print(f'  Lignes après nettoyage : {len(df_liv):,}')
print(f'  Poids corrigés         : 2')
print(f'  CSAT mis à NULL        : 3')
print(f'  Délais recalculés      : 4')
print(f'  Transporteur suspendu  : marqué (colonne transporteur_actif)')
print(f'  Nulls restants delai_jours : {df_liv["delai_jours"].isna().sum()}')
print(f'  CSAT non renseigné     : {df_liv["csat"].isna().sum():,} ({df_liv["csat"].isna().mean()*100:.1f}%)')

---
## Étape 3 — Feature Engineering

### MÉTHODE — Pourquoi créer des features ?
Le modèle ML de NB5 ne peut pas utiliser les colonnes brutes directement :
- `transporteur_id` est un identifiant — le modèle ne peut pas apprendre de `TRP12` sans contexte de performance
- `pays_origine` est une chaîne — il faut l'encoder ou extraire sa sémantique (risque douanier)
- `date_creation` est une date — il faut en extraire les composantes temporelles

**Objectif :** créer des features que le modèle peut interpréter et qui capturent la connaissance métier.

### 3.1 — Variables temporelles

> **MÉTHODE — Pourquoi le mois en sin/cos ?**
>
> Le mois est cyclique : décembre (12) est proche de janvier (1) dans la réalité, mais `12 - 1 = 11` numériquement. Encoder en `sin(2π × mois/12)` et `cos(2π × mois/12)` préserve cette proximité cyclique — le modèle comprendra que décembre et janvier sont dans le même pic saisonnier.

In [ ]:
# Variables temporelles existantes — à valider
print('Distribution mois :', df_liv['mois'].value_counts().sort_index().to_dict())

# TODO : créer mois_sin = sin(2π × mois / 12)
# TODO : créer mois_cos = cos(2π × mois / 12)
df_liv['mois_sin'] = # TODO
df_liv['mois_cos'] = # TODO

# TODO : créer heure_enlevement depuis date_enlevement.dt.hour
df_liv['heure_enlevement'] = # TODO

# TODO : créer est_fin_mois = 1 si jour >= 28
df_liv['est_fin_mois'] = # TODO

# TODO : créer delai_prep_heures = (date_enlevement - date_creation) en heures
# Indice : .dt.total_seconds() / 3600, puis .clip(lower=0)
df_liv['delai_prep_heures'] = # TODO

print('Variables temporelles créées ✅')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 3.2 — Features de performance historique

### MÉTHODE — Anti-leakage sur les taux historiques
> Le taux de breach d'un transporteur calculé sur **toutes** les données serait du leakage : le modèle connaîtrait déjà les performances futures. On calcule les taux sur les données **passées uniquement** — pour chaque livraison, on n'utilise que les livraisons antérieures.
>
> En NB5, la coupure temporelle gérera ce problème proprement. Ici, pour la simplicité, on calcule les taux sur l'ensemble et on note qu'en production le calcul sera fenêtré.

In [ ]:
# Taux breach historique par transporteur
# TODO : groupby transporteur_id → mean(sla_breach) → rename taux_breach_transporteur
taux_breach_trp = # TODO
df_liv = df_liv.merge(taux_breach_trp, on='transporteur_id', how='left')

# Taux breach historique par corridor
# TODO : groupby [pays_origine, pays_destination] → mean(sla_breach)
taux_breach_corr = # TODO
df_liv = df_liv.merge(taux_breach_corr, on=['pays_origine','pays_destination'], how='left')

# Taux breach par entrepôt d'origine
# TODO : groupby entrepot_origine_id → mean(sla_breach)
taux_breach_ent = # TODO
df_liv = df_liv.merge(taux_breach_ent, on='entrepot_origine_id', how='left')

print('Features historiques créées ✅')

### 3.3 — Features logistiques métier

In [ ]:
# Enrichissement depuis routes
df_liv = df_liv.merge(
    df_rte[['route_id','distance_km','duree_standard_jours',
            'risque_douanier','nb_points_controle']],
    on='route_id', how='left'
)

# TODO : ratio_duree_vs_standard = duree_reelle_jours / duree_standard_jours
df_liv['ratio_duree_vs_standard'] = # TODO

# TODO : delai_relatif_sla = duree_reelle_jours / sla_jours_contractuel
df_liv['delai_relatif_sla'] = # TODO

# TODO : encoder risque_douanier → risque_douanier_num
# Faible=0, Moyen=1, Élevé=2
risque_map = {'Faible': 0, 'Moyen': 1, 'Élevé': 2}
df_liv['risque_douanier_num'] = # TODO

# TODO : encoder priorite → priorite_num (Standard=0, Express=1, Urgent=2)
priorite_map = {'Standard': 0, 'Express': 1, 'Urgent': 2}
df_liv['priorite_num'] = # TODO

# TODO : densite_kg_m3 = poids_kg / volume_m3 (gérer les zéros)
df_liv['densite_kg_m3'] = # TODO

print('Features logistiques créées ✅')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

### 3.4 — Enrichissement depuis les référentiels

In [ ]:
# Joindre les caractéristiques du transporteur
df_liv = df_liv.merge(
    df_trp[['transporteur_id','mode_transport','taux_livraison_temps','tarif_moyen_fcfa_100kg']],
    on='transporteur_id', how='left'
)

# Encodage mode transport
mode_map = {'Routier': 0, 'Multimodal': 1, 'Maritime': 2, 'Aérien': 3}
df_liv['mode_transport_num'] = df_liv['mode_transport'].map(mode_map)

# Joindre taux ponctualité entrepôt d'origine
df_liv = df_liv.merge(
    df_ent[['entrepot_id','taux_ponctualite']].rename(
        columns={'entrepot_id':'entrepot_origine_id','taux_ponctualite':'ponctualite_entrepot_origine'}
    ),
    on='entrepot_origine_id', how='left'
)

# Joindre SLA et pénalité client
df_liv = df_liv.merge(
    df_cli[['client_id','segment','penalite_retard_pct']],
    on='client_id', how='left'
)

# Encodage segment client
segment_map = {'PME': 0, 'Administration': 1, 'Grand Compte': 2}
df_liv['segment_num'] = df_liv['segment'].map(segment_map)

print('Enrichissements référentiels créés :')
print(f'  taux_livraison_temps            : mean={df_liv["taux_livraison_temps"].mean():.3f}')
print(f'  ponctualite_entrepot_origine    : mean={df_liv["ponctualite_entrepot_origine"].mean():.3f}')
print(f'  penalite_retard_pct             : mean={df_liv["penalite_retard_pct"].mean():.2f}')
print(f'  segment_num distribution        : {df_liv["segment_num"].value_counts().to_dict()}')

---
## Étape 4 — Validation de la variable cible

### MÉTHODE — Définition de `livraison_en_retard`
La variable cible est déjà dans le CSV mais on la revalide ici pour s'assurer de sa cohérence avec les colonnes nettoyées :

```
livraison_en_retard = 1  si  sla_breach = 1  OR  in_backlog = 1  OR  escalade = 1
                     = 0  sinon
```

> **MÉTIER :** Cette définition inclusive capture trois types de risque business :
> - `sla_breach` : impact financier immédiat (pénalités)
> - `escalade` : impact relationnel (client insatisfait qui remonte au management)
> - `in_backlog` : risque latent (livraison bloquée qui va breacher)

In [ ]:
# Revalider la variable cible après nettoyage
# TODO : recalculer livraison_en_retard = sla_breach OR in_backlog OR escalade
df_liv['livraison_en_retard_check'] = (
    # TODO
).astype(int)

diff = (df_liv['livraison_en_retard'] != df_liv['livraison_en_retard_check']).sum()
print(f'Incohérences : {diff}')
if diff == 0:
    print('✅ Variable cible cohérente')
    df_liv.drop(columns=['livraison_en_retard_check'], inplace=True)

vc = df_liv['livraison_en_retard'].value_counts()
print(vc)
print(f'Taux classe positive : {vc[1]/len(df_liv)*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df_liv['livraison_en_retard'].value_counts()
bars = ax.bar(['À l\'heure (0)', 'En retard (1)'], counts.values,
              color=[COLORS['secondary'], COLORS['danger']])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(df_liv)*100:.1f}%)',
            ha='center', fontweight='bold')
ax.set_title('Distribution de la variable cible livraison_en_retard', fontweight='bold')
ax.set_ylabel('Nb livraisons')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_target_distribution.png')

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 5 — Sélection des features et export

### MÉTHODE — Pourquoi sélectionner les colonnes ?
On exporte uniquement les colonnes nécessaires pour le ML et le dashboard. Les colonnes sources redondantes (`date_livraison_reelle`, `risque_douanier` texte, `mode_transport` texte) sont exclues :
- **Leakage :** `date_livraison_reelle` contient implicitement le résultat de la livraison
- **Redondance :** `risque_douanier_num` remplace `risque_douanier` pour le ML
- **Poids :** un DataFrame allégé est plus rapide à charger en NB3, NB5 et Power BI

In [ ]:
COLS_ANALYTICS = [
    # Identifiants
    'livraison_id', 'client_id', 'transporteur_id',
    'entrepot_origine_id', 'entrepot_dest_id', 'route_id',
    # Géographie
    'pays_origine', 'pays_destination',
    # Dates et temporel
    'date_creation', 'date_enlevement', 'date_livraison_prevue',
    'annee', 'mois', 'jour_semaine', 'est_weekend',
    'est_fin_mois', 'heure_enlevement', 'delai_prep_heures',
    'mois_sin', 'mois_cos',
    # Logistique
    'type_colis', 'poids_kg', 'volume_m3', 'densite_kg_m3',
    'priorite', 'priorite_num',
    'sla_jours_contractuel', 'duree_reelle_jours',
    'distance_km', 'duree_standard_jours', 'nb_points_controle',
    'ratio_duree_vs_standard', 'delai_relatif_sla',
    # Performance historique
    'taux_breach_transporteur', 'taux_breach_corridor',
    'taux_breach_entrepot_origine', 'taux_livraison_temps',
    'ponctualite_entrepot_origine',
    # Référentiels encodés
    'risque_douanier', 'risque_douanier_num',
    'mode_transport', 'mode_transport_num',
    'segment', 'segment_num',
    'penalite_retard_pct', 'transporteur_actif',
    # Résultats
    'statut', 'sla_breach', 'escalade', 'in_backlog', 'incident',
    'delai_jours', 'csat', 'cout_transport_fcfa', 'penalite_fcfa',
    # Variable cible ML
    'livraison_en_retard',
]

df_analytics = df_liv[COLS_ANALYTICS].copy()
df_analytics.to_csv(f'{SAVE_PATH}logitrack_analytics.csv', index=False)

print(f'✅ logitrack_analytics.csv exporté')
print(f'   Dimensions     : {df_analytics.shape}')
print(f'   Nulls restants : {df_analytics.isnull().sum().sum()}')
print(f'   Localisation   : {SAVE_PATH}')
print(f'\nAperçu des 5 premières colonnes :')
print(df_analytics.head(3).to_string())

---
## Bilan du Notebook 2

### Corrections appliquées

| Action | Nb | Méthode |
|---|---|---|
| Poids aberrants corrigés | **2** | Médiane par type_colis |
| CSAT hors plage → NULL | **3** | Valeur inconnue > valeur fausse |
| Délais aberrants recalculés | **4** | `duree_reelle - sla_contractuel` |
| Transporteur suspendu marqué | **1** | Colonne `transporteur_actif` |

### Features créées

| Feature | Type | Rôle ML |
|---|---|---|
| `mois_sin` / `mois_cos` | Float | Saisonnalité cyclique |
| `est_fin_mois` | Binaire | Pic de fin de période |
| `delai_prep_heures` | Float | Efficacité logistique à l'origine |
| `taux_breach_transporteur` | Float | Performance historique partenaire |
| `taux_breach_corridor` | Float | Fiabilité de la route |
| `taux_breach_entrepot_origine` | Float | Goulot d'étranglement entrepôt |
| `delai_relatif_sla` | Float | Position vs SLA (>1 = en retard) |
| `ratio_duree_vs_standard` | Float | Écart durée réelle vs théorique |
| `risque_douanier_num` | Ordinal | 0=Faible, 1=Moyen, 2=Élevé |
| `priorite_num` | Ordinal | 0=Standard, 1=Express, 2=Urgent |
| `ponctualite_entrepot_origine` | Float | Fiabilité du hub de départ |

**Fichier exporté :** `logitrack_analytics.csv` — disponible dans `SAVE_PATH`

**Pour le NB3 :** charger `logitrack_analytics.csv` avec DuckDB pour les analyses SQL avancées — corridors, transporteurs, LAG mensuel, heatmap volume.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.